In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

project_path = "/content/drive/MyDrive/olist-late-delivery-prediction"

folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "sql",
    "reports",
    "models",
    "src"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folders created successfully!")

In [ ]:
import zipfile
import glob

raw_path = os.path.join(project_path, "data", "raw")
extract_path = os.path.join(raw_path, "extracted")

os.makedirs(extract_path, exist_ok=True)

zip_files = glob.glob(os.path.join(raw_path, "*.zip"))

print("ZIP files found:", zip_files)

In [ ]:
import os
import glob

raw_path = os.path.join(project_path, "data", "raw")
extract_path = os.path.join(raw_path, "extracted")
os.makedirs(extract_path, exist_ok=True)

zip_files = glob.glob(os.path.join(raw_path, "*.zip"))

if not zip_files:
    print(f"Error: No zip files found in {raw_path}. Please upload your dataset zip file here.")
else:
    zip_path = zip_files[0]
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Dataset extracted successfully!")

In [ ]:
csv_files = glob.glob(
    os.path.join(extract_path, "**", "*.csv"),
    recursive=True
)

print("Total CSV files:", len(csv_files))

for file in csv_files:
    print(os.path.basename(file))

In [ ]:
import os
import glob
import pandas as pd

# Re-locate your extracted dataset path
project_path = "/content/drive/MyDrive/olist-late-delivery-prediction"
extract_path = os.path.join(project_path, "data", "raw", "extracted")

# Find and load all CSV files back into memory
csv_files = glob.glob(os.path.join(extract_path, "**", "*.csv"), recursive=True)

data = {}
for file in csv_files:
    name = os.path.basename(file).replace(".csv", "")
    data[name] = pd.read_csv(file, low_memory=False)

print("Successfully re-loaded", len(data), "datasets into memory!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

In [ ]:
data = {}

for file in csv_files:
    name = os.path.basename(file).replace(".csv", "")

    data[name] = pd.read_csv(file, low_memory=False)

    print(name, ":", data[name].shape)

In [ ]:
orders = data["olist_orders_dataset"]

orders.head()

In [ ]:
orders.info()

In [ ]:
orders.describe(include="all")

In [ ]:
missing_values = orders.isnull().sum()

missing_percentage = (
    orders.isnull().sum() / len(orders)
) * 100

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

missing_summary.sort_values(
    by="Missing Percentage",
    ascending=False
)

In [ ]:
print("Total duplicate rows:", orders.duplicated().sum())

print(
    "Duplicate order IDs:",
    orders["order_id"].duplicated().sum()
)

In [ ]:
orders["order_status"].value_counts()

In [ ]:
status_percentage = (
    orders["order_status"].value_counts(normalize=True) * 100
)

print(status_percentage.round(2))

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": orders.isnull().sum(),
    "Missing Percentage": orders.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(
    by="Missing Percentage",
    ascending=False
)

missing_summary

In [ ]:
# Check for completely duplicated rows
print("Duplicate rows:", orders.duplicated().sum())

# Check whether any order ID appears more than once
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

In [ ]:
# Display the shape and columns of every dataset

for name, df in data.items():
    print("\n" + "=" * 60)
    print("DATASET:", name)
    print("Rows and columns:", df.shape)
    print("\nColumn names:")
    print(df.columns.tolist())

In [ ]:
# Check missing values and duplicate rows in every dataset

for name, df in data.items():

    print("\n" + "=" * 60)
    print("DATASET:", name)

    print("Total rows:", len(df))
    print("Duplicate rows:", df.duplicated().sum())
    print("Total missing values:", df.isnull().sum().sum())

    print("\nMissing values by column:")
    print(df.isnull().sum())

In [ ]:
# Check whether important IDs are unique

# Check whether important IDs are unique
unique_checks = {
    "olist_orders_dataset": ["order_id"],
    "olist_customers_dataset": ["customer_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_products_dataset": ["product_id"],
    "olist_order_reviews_dataset": ["review_id"],
    "product_category_name_translation": ["product_category_name"]
}

for table_name, columns in unique_checks.items():
    df = data[table_name]
    for column in columns:
        print(
            table_name,
            column,
            "Unique:",
            df[column].is_unique
        )

In [ ]:
reviews = data["olist_order_reviews_dataset"]

# Count repeated review IDs
repeated_reviews = reviews[
    reviews["review_id"].duplicated(keep=False)
]

print("Total rows with repeated review IDs:", len(repeated_reviews))
print("Number of distinct repeated review IDs:",
      repeated_reviews["review_id"].nunique())

# Display examples
repeated_reviews.sort_values("review_id").head(10)

In [ ]:
geolocation = data["olist_geolocation_dataset"]

print("Total geolocation rows:", len(geolocation))
print("Exact duplicate rows:", geolocation.duplicated().sum())

# Check how many ZIP prefixes appear more than once
print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

# Examine one ZIP prefix with multiple records
zip_counts = geolocation["geolocation_zip_code_prefix"].value_counts()

print("\nMost frequent ZIP prefixes:")
print(zip_counts.head(10))

In [ ]:
reviews = data["olist_order_reviews_dataset"]

print(
    "Duplicate review_id + order_id pairs:",
    reviews.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

orders_per_review = reviews.groupby("review_id")["order_id"].nunique()

print(
    "Review IDs associated with multiple orders:",
    (orders_per_review > 1).sum()
)

print(
    "Review ID/order pairs with multiple records:",
    (reviews.groupby(["review_id", "order_id"]).size() > 1).sum()
)

In [ ]:
geolocation = data["olist_geolocation_dataset"]

geo_clean = geolocation.drop_duplicates().copy()

print("Original rows:", len(geolocation))
print("Rows after exact deduplication:", len(geo_clean))

states_per_zip = geo_clean.groupby(
    "geolocation_zip_code_prefix"
)["geolocation_state"].nunique()

print(
    "ZIP prefixes associated with multiple states:",
    (states_per_zip > 1).sum()
)

print("\nMost frequent ZIP prefixes after cleaning:")
print(
    geo_clean["geolocation_zip_code_prefix"]
    .value_counts()
    .head(10)
)

In [ ]:
geolocation = data["olist_geolocation_dataset"]
geo_clean = geolocation.drop_duplicates().copy()

print("Original rows:", len(geolocation))
print("Rows after exact deduplication:", len(geo_clean))

states_per_zip = geo_clean.groupby("geolocation_zip_code_prefix")["geolocation_state"].nunique()
print("ZIP prefixes associated with multiple states:", (states_per_zip > 1).sum())

print("\nMost frequent ZIP prefixes after cleaning:")
print(geo_clean["geolocation_zip_code_prefix"].value_counts().head(10))

In [ ]:
# Load the required tables
orders = data["olist_orders_dataset"]
customers = data["olist_customers_dataset"]
items = data["olist_order_items_dataset"]
products = data["olist_products_dataset"]
sellers = data["olist_sellers_dataset"]
payments = data["olist_order_payments_dataset"]
reviews = data["olist_order_reviews_dataset"]

# Define relationships to validate
relationships = [
    ("Orders -> Customers",
     orders, "customer_id", customers, "customer_id"),

    ("Order Items -> Orders",
     items, "order_id", orders, "order_id"),

    ("Order Items -> Products",
     items, "product_id", products, "product_id"),

    ("Order Items -> Sellers",
     items, "seller_id", sellers, "seller_id"),

    ("Payments -> Orders",
     payments, "order_id", orders, "order_id"),

    ("Reviews -> Orders",
     reviews, "order_id", orders, "order_id")
]

# Check unmatched IDs
for name, child, child_col, parent, parent_col in relationships:
    unmatched = (~child[child_col].isin(parent[parent_col])).sum()

    print(name)
    print("Unmatched records:", unmatched)
    print("-" * 45)

In [ ]:
print(
    "Duplicate order-item combinations:",
    items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "Duplicate payment-sequential combinations:",
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

In [ ]:
!pip install -q psycopg2-binary

In [ ]:
from getpass import getpass
DATABASE_URL = getpass("Enter database connection URL: ")


In [ ]:
import psycopg2

try:
    conn = psycopg2.connect(DATABASE_URL)
    cursor = conn.cursor()

    cursor.execute("SELECT version();")
    version = cursor.fetchone()

    cursor.execute("SELECT current_database();")
    database_name = cursor.fetchone()

    print("Connection successful!")
    print("Database:", database_name[0])
    print("PostgreSQL version:", version[0])

    cursor.close()

except Exception as e:
    print("Connection failed:")
    print(e)

In [ ]:
print("Connection string entered:", bool(DATABASE_URL))
print("Connection string starts with:", DATABASE_URL[:15] if DATABASE_URL else "EMPTY")

In [ ]:
# Securely request the database URL without displaying it
DATABASE_URL = getpass("Enter database connection URL: ")

assert DATABASE_URL.startswith(("postgresql://", "postgres://")), \
    "Please enter a valid PostgreSQL connection string."

print("Connection string captured successfully!")

In [ ]:
import psycopg2

conn = psycopg2.connect(DATABASE_URL)
cursor = conn.cursor()

cursor.execute("SELECT current_database(), version();")
result = cursor.fetchone()

print("Connected successfully!")
print("Database:", result[0])
print("PostgreSQL:", result[1].split(",")[0])

cursor.close()

In [ ]:
!pip install -q sqlalchemy

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True
)

with engine.connect() as connection:
    result = connection.execute(text("SELECT current_database();"))
    print("Connected database:", result.fetchone()[0])

In [ ]:
!pip install -q psycopg

In [ ]:
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS raw;"))

print("Raw schema created successfully!")

In [ ]:
print("Number of datasets:", len(data))

for name, df in data.items():
    print(f"{name}: {df.shape}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob

project_path = "/content/drive/MyDrive/olist-late-delivery-prediction"

csv_files = glob.glob(
    os.path.join(project_path, "data", "raw", "extracted", "**", "*.csv"),
    recursive=True
)

print("CSV files found:", len(csv_files))

for file in csv_files:
    print(os.path.basename(file))

In [ ]:
import pandas as pd
import numpy as np

data = {}

for file in csv_files:
    name = os.path.basename(file).replace(".csv", "")
    data[name] = pd.read_csv(file, low_memory=False)

print("Datasets loaded:", len(data))

for name, df in data.items():
    print(f"{name}: {df.shape}")

In [ ]:
table_mapping = {
    "olist_customers_dataset": "customers",
    "olist_geolocation_dataset": "geolocation",
    "olist_order_items_dataset": "order_items",
    "olist_order_payments_dataset": "order_payments",
    "olist_order_reviews_dataset": "order_reviews",
    "olist_orders_dataset": "orders",
    "olist_products_dataset": "products",
    "olist_sellers_dataset": "sellers",
    "product_category_name_translation": "category_translation"
}

print("Tables to import:", len(table_mapping))

for dataset_name, table_name in table_mapping.items():
    print(dataset_name, "-> raw." + table_name)

In [ ]:
for dataset_name, table_name in table_mapping.items():

    df = data[dataset_name]

    print(f"Importing {table_name}...")

    df.to_sql(
        name=table_name,
        con=engine,
        schema="raw",
        if_exists="replace",
        index=False,
        chunksize=5000
    )

    print(f"Imported {len(df):,} rows into raw.{table_name}")

print("\nAll nine datasets imported!")

In [ ]:
from sqlalchemy import text

with engine.connect() as connection:

    result = connection.execute(text("""
        SELECT
            table_name
        FROM information_schema.tables
        WHERE table_schema = 'raw'
        ORDER BY table_name;
    """))

    print("Tables inside raw schema:")

    for row in result:
        print(row[0])

In [ ]:
with engine.connect() as connection:

    for table_name in table_mapping.values():

        query = text(f'SELECT COUNT(*) FROM raw."{table_name}"')

        count = connection.execute(query).scalar()

        print(f"{table_name}: {count:,} rows")

In [ ]:
query = """
SELECT
    order_status,
    COUNT(*) AS total_orders
FROM raw.orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM raw.orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    c.customer_city,
    c.customer_state
FROM raw.orders AS o
INNER JOIN raw.customers AS c
    ON o.customer_id = c.customer_id
LIMIT 10;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)


In [ ]:
query = """
SELECT COUNT(*) AS joined_orders
FROM raw.orders AS o
INNER JOIN raw.customers AS c
    ON o.customer_id = c.customer_id;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

print("Orders after JOIN:", result.iloc[0, 0])

In [ ]:
query = """
SELECT
    c.customer_state,
    COUNT(o.order_id) AS total_orders
FROM raw.orders AS o
INNER JOIN raw.customers AS c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_orders DESC
LIMIT 10;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    order_id,
    COUNT(*) AS number_of_items,
    SUM(price) AS total_product_value,
    SUM(freight_value) AS total_freight
FROM raw.order_items
GROUP BY order_id
ORDER BY number_of_items DESC
LIMIT 10;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    (SELECT COUNT(*) FROM raw.orders) AS order_count,
    (SELECT COUNT(*) FROM raw.order_items) AS item_count,
    (
        SELECT COUNT(*)
        FROM raw.orders o
        INNER JOIN raw.order_items oi
            ON o.order_id = oi.order_id
    ) AS joined_row_count;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE SCHEMA IF NOT EXISTS analytics;
    """))

print("Analytics schema created!")

In [ ]:
query = """
CREATE OR REPLACE VIEW analytics.order_item_summary AS
SELECT
    order_id,
    COUNT(*) AS total_items,
    COUNT(DISTINCT product_id) AS unique_products,
    COUNT(DISTINCT seller_id) AS unique_sellers,
    SUM(price) AS total_product_value,
    SUM(freight_value) AS total_freight,
    AVG(price) AS average_item_price
FROM raw.order_items
GROUP BY order_id;
"""

with engine.begin() as connection:
    connection.execute(text(query))

print("Order item summary view created!")

In [ ]:
!pip install -q psycopg

In [ ]:
import os
import glob
import pandas as pd
from sqlalchemy import create_engine, text

# Make sure there are no spaces in the URL string
DATABASE_URL = getpass("Enter your Neon Database URL: ")

# Re-create the engine
engine = create_engine(DATABASE_URL, pool_pre_ping=True)
print("Engine re-initialized successfully!")

In [ ]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE SCHEMA IF NOT EXISTS analytics;
    """))
print("Analytics schema created!")

In [ ]:
query = """
CREATE OR REPLACE VIEW analytics.order_item_summary AS
SELECT
    order_id,
    COUNT(*) AS total_items,
    COUNT(DISTINCT product_id) AS unique_products,
    COUNT(DISTINCT seller_id) AS unique_sellers,
    SUM(price) AS total_product_value,
    SUM(freight_value) AS total_freight,
    AVG(price) AS average_item_price
FROM raw.order_items
GROUP BY order_id;
"""

with engine.begin() as connection:
    connection.execute(text(query))

print("Order item summary view created!")

In [ ]:
query = """
SELECT *
FROM analytics.order_item_summary
ORDER BY total_items DESC
LIMIT 10;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    COUNT(*) AS total_orders,
    COUNT(oi.order_id) AS orders_with_items,
    COUNT(*) - COUNT(oi.order_id) AS orders_without_items
FROM raw.orders AS o
LEFT JOIN analytics.order_item_summary AS oi
    ON o.order_id = oi.order_id;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM (
    SELECT
        o.order_id,
        oi.total_items,
        oi.total_product_value,
        oi.total_freight
    FROM raw.orders AS o
    LEFT JOIN analytics.order_item_summary AS oi
        ON o.order_id = oi.order_id
) AS combined;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    payment_type,
    COUNT(*) AS payment_records,
    ROUND(SUM(payment_value)::numeric, 2) AS total_payment_value
FROM raw.order_payments
GROUP BY payment_type
ORDER BY payment_records DESC;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
CREATE OR REPLACE VIEW analytics.payment_summary AS
SELECT
    order_id,
    COUNT(*) AS payment_records,
    SUM(payment_value) AS total_payment_value,
    SUM(payment_installments) AS total_installments,
    MAX(payment_installments) AS max_installments
FROM raw.order_payments
GROUP BY order_id;
"""

with engine.begin() as connection:
    connection.execute(text(query))

print("Payment summary view created!")

In [ ]:
query = """
SELECT
    payment_type,
    COUNT(*) AS payment_records,
    ROUND(SUM(payment_value)::numeric, 2) AS total_payment_value
FROM raw.order_payments
GROUP BY payment_type
ORDER BY payment_records DESC;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    review_score,
    COUNT(*) AS total_reviews,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM raw.order_reviews
GROUP BY review_score
ORDER BY review_score;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT *
FROM analytics.payment_summary
LIMIT 10;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    COUNT(*) AS summary_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM analytics.payment_summary;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    COUNT(*) AS total_orders,
    COUNT(ps.order_id) AS orders_with_payments,
    COUNT(*) - COUNT(ps.order_id) AS orders_without_payments
FROM raw.orders o
LEFT JOIN analytics.payment_summary ps
    ON o.order_id = ps.order_id;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    review_score,
    COUNT(*) AS total_reviews,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM raw.order_reviews
GROUP BY review_score
ORDER BY review_score;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
CREATE OR REPLACE VIEW analytics.review_summary AS
SELECT
    order_id,
    COUNT(*) AS review_count,
    AVG(review_score) AS average_review_score,
    MIN(review_score) AS minimum_review_score,
    MAX(review_score) AS maximum_review_score
FROM raw.order_reviews
GROUP BY order_id;
"""

with engine.begin() as connection:
    connection.execute(text(query))

print("Review summary view created!")

In [ ]:
query = """
SELECT *
FROM analytics.review_summary
LIMIT 10;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    COUNT(*) AS summary_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM analytics.review_summary;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
CREATE OR REPLACE VIEW analytics.order_level_base AS

WITH typed_orders AS (
    SELECT
        order_id,
        customer_id,
        order_status,
        order_purchase_timestamp::timestamp AS purchase_timestamp,
        order_delivered_customer_date::timestamp AS actual_delivery_timestamp,
        order_estimated_delivery_date::timestamp AS estimated_delivery_timestamp
    FROM raw.orders
)

SELECT
    o.order_id,
    o.customer_id,

    -- Customer information
    c.customer_city,
    c.customer_state,

    -- Information available around order placement
    o.purchase_timestamp,
    oi.total_items,
    oi.unique_products,
    oi.unique_sellers,
    oi.total_product_value,
    oi.total_freight,
    oi.average_item_price,

    ps.payment_records,
    ps.total_payment_value,
    ps.max_installments,

    -- Delivery outcome information
    o.actual_delivery_timestamp,
    o.estimated_delivery_timestamp,

    CASE
        WHEN o.actual_delivery_timestamp > o.estimated_delivery_timestamp
        THEN 1
        ELSE 0
    END AS is_late,

    ROUND(
        (EXTRACT(EPOCH FROM (
            o.actual_delivery_timestamp - o.purchase_timestamp
        )) / 86400.0)::numeric,
        2
    ) AS actual_delivery_days

FROM typed_orders o

LEFT JOIN raw.customers c
    ON o.customer_id = c.customer_id

LEFT JOIN analytics.order_item_summary oi
    ON o.order_id = oi.order_id

LEFT JOIN analytics.payment_summary ps
    ON o.order_id = ps.order_id

WHERE
    o.order_status = 'delivered'
    AND o.actual_delivery_timestamp IS NOT NULL
    AND o.estimated_delivery_timestamp IS NOT NULL;
"""

with engine.begin() as connection:
    connection.execute(text(query))

print("Order-level analytical view created!")

In [ ]:
query = """
SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(is_late) AS late_orders,
    COUNT(*) - SUM(is_late) AS on_time_orders
FROM analytics.order_level_base;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    is_late,
    COUNT(*) AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM analytics.order_level_base
GROUP BY is_late
ORDER BY is_late;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM analytics.order_level_base;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
SELECT *
FROM analytics.order_level_base
LIMIT 10;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

In [ ]:
query = """
CREATE OR REPLACE VIEW analytics.order_level_base AS

WITH typed_orders AS (
    SELECT
        order_id,
        customer_id,
        order_status,
        order_purchase_timestamp::timestamp AS purchase_timestamp,
        order_delivered_customer_date::timestamp AS actual_delivery_timestamp,
        order_estimated_delivery_date::timestamp AS estimated_delivery_timestamp
    FROM raw.orders
)

SELECT
    o.order_id,
    o.customer_id,
    c.customer_city,
    c.customer_state,
    o.purchase_timestamp,
    oi.total_items,
    oi.unique_products,
    oi.unique_sellers,
    oi.total_product_value,
    oi.total_freight,
    oi.average_item_price,
    ps.payment_records,
    ps.total_payment_value,
    ps.max_installments,
    o.actual_delivery_timestamp,
    o.estimated_delivery_timestamp,

    CASE
        WHEN o.actual_delivery_timestamp::date >
             o.estimated_delivery_timestamp::date
        THEN 1
        ELSE 0
    END AS is_late,

    ROUND(
        (EXTRACT(EPOCH FROM (
            o.actual_delivery_timestamp - o.purchase_timestamp
        )) / 86400.0)::numeric,
        2
    ) AS actual_delivery_days

FROM typed_orders o
LEFT JOIN raw.customers c
    ON o.customer_id = c.customer_id
LEFT JOIN analytics.order_item_summary oi
    ON o.order_id = oi.order_id
LEFT JOIN analytics.payment_summary ps
    ON o.order_id = ps.order_id

WHERE
    o.order_status = 'delivered'
    AND o.actual_delivery_timestamp IS NOT NULL
    AND o.estimated_delivery_timestamp IS NOT NULL;
"""

with engine.begin() as connection:
    connection.execute(text(query))

print("Corrected order-level view created!")

In [ ]:
query = """
SELECT
    COUNT(*) AS total_orders,
    SUM(is_late) AS late_orders,
    COUNT(*) - SUM(is_late) AS on_time_orders,
    ROUND(AVG(is_late::numeric) * 100, 2) AS late_percentage
FROM analytics.order_level_base;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)
display(result)

In [ ]:
query = """
SELECT
    COUNT(*) AS total_orders,
    SUM(
        CASE
            WHEN actual_delivery_timestamp > estimated_delivery_timestamp
            THEN 1 ELSE 0
        END
    ) AS late_orders,
    SUM(
        CASE
            WHEN actual_delivery_timestamp <= estimated_delivery_timestamp
            THEN 1 ELSE 0
        END
    ) AS on_time_orders,
    SUM(
        CASE
            WHEN actual_delivery_timestamp IS NULL
              OR estimated_delivery_timestamp IS NULL
            THEN 1 ELSE 0
        END
    ) AS missing_dates
FROM analytics.order_level_base;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)
display(result)

In [ ]:
query = """
SELECT
    customer_state,
    COUNT(*) AS total_orders,
    SUM(is_late) AS late_orders,
    ROUND(
        SUM(is_late) * 100.0 / COUNT(*),
        2
    ) AS late_percentage
FROM analytics.order_level_base
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY late_percentage DESC;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)
display(result)

In [ ]:
query = """
SELECT
    CASE
        WHEN total_items = 1 THEN '1 item'
        WHEN total_items = 2 THEN '2 items'
        WHEN total_items = 3 THEN '3 items'
        WHEN total_items = 4 THEN '4 items'
        ELSE '5+ items'
    END AS order_size,
    COUNT(*) AS total_orders,
    SUM(is_late) AS late_orders,
    ROUND(
        SUM(is_late) * 100.0 / COUNT(*),
        2
    ) AS late_percentage
FROM analytics.order_level_base
GROUP BY
    CASE
        WHEN total_items = 1 THEN '1 item'
        WHEN total_items = 2 THEN '2 items'
        WHEN total_items = 3 THEN '3 items'
        WHEN total_items = 4 THEN '4 items'
        ELSE '5+ items'
    END
ORDER BY MIN(total_items);
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)
display(result)

In [ ]:
query = """
SELECT
    CASE
        WHEN total_freight < 20 THEN 'Low freight (<20)'
        WHEN total_freight < 50 THEN 'Medium freight (20-50)'
        ELSE 'High freight (50+)'
    END AS freight_category,
    COUNT(*) AS total_orders,
    SUM(is_late) AS late_orders,
    ROUND(
        SUM(is_late) * 100.0 / COUNT(*),
        2
    ) AS late_percentage
FROM analytics.order_level_base
GROUP BY
    CASE
        WHEN total_freight < 20 THEN 'Low freight (<20)'
        WHEN total_freight < 50 THEN 'Medium freight (20-50)'
        ELSE 'High freight (50+)'
    END
ORDER BY MIN(total_freight);
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)
display(result)

In [ ]:
query = """
SELECT
    TO_CHAR(
        DATE_TRUNC('month', purchase_timestamp),
        'YYYY-MM'
    ) AS purchase_month,
    COUNT(*) AS total_orders,
    SUM(is_late) AS late_orders,
    ROUND(
        SUM(is_late) * 100.0 / COUNT(*),
        2
    ) AS late_percentage
FROM analytics.order_level_base
GROUP BY DATE_TRUNC('month', purchase_timestamp)
ORDER BY DATE_TRUNC('month', purchase_timestamp);
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)
display(result)

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Define project path
project_path = "/content/drive/MyDrive/olist-late-delivery-prediction"

# Retrieve the four EDA results from PostgreSQL
state_df = pd.read_sql(text("""
    SELECT customer_state,
           COUNT(*) AS total_orders,
           ROUND(SUM(is_late) * 100.0 / COUNT(*), 2)
               AS late_percentage
    FROM analytics.order_level_base
    GROUP BY customer_state
    HAVING COUNT(*) >= 100
    ORDER BY late_percentage DESC
    LIMIT 10
"""), engine)

size_df = pd.read_sql(text("""
    SELECT
        CASE
            WHEN total_items = 1 THEN '1 item'
            WHEN total_items = 2 THEN '2 items'
            WHEN total_items = 3 THEN '3 items'
            WHEN total_items = 4 THEN '4 items'
            ELSE '5+ items'
        END AS order_size,
        COUNT(*) AS total_orders,
        ROUND(SUM(is_late) * 100.0 / COUNT(*), 2)
            AS late_percentage
    FROM analytics.order_level_base
    GROUP BY 1
"""), engine)

freight_df = pd.read_sql(text("""
    SELECT
        CASE
            WHEN total_freight < 20 THEN 'Low (<20)'
            WHEN total_freight < 50 THEN 'Medium (20-50)'
            ELSE 'High (50+)'
        END AS freight_category,
        COUNT(*) AS total_orders,
        ROUND(SUM(is_late) * 100.0 / COUNT(*), 2)
            AS late_percentage
    FROM analytics.order_level_base
    GROUP BY 1
"""), engine)

monthly_df = pd.read_sql(text("""
    SELECT
        DATE_TRUNC('month', purchase_timestamp) AS month,
        COUNT(*) AS total_orders,
        ROUND(SUM(is_late) * 100.0 / COUNT(*), 2)
            AS late_percentage
    FROM analytics.order_level_base
    GROUP BY 1
    HAVING COUNT(*) >= 100
    ORDER BY 1
"""), engine)

# Create a 2x2 dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Late delivery by state
sns.barplot(
    data=state_df,
    x="late_percentage",
    y="customer_state",
    ax=axes[0, 0]
)
axes[0, 0].set_title("Top 10 States by Late Delivery Rate")
axes[0, 0].set_xlabel("Late Delivery (%)")
axes[0, 0].set_ylabel("State")

# 2. Late delivery by order size
size_order = ["1 item", "2 items", "3 items", "4 items", "5+ items"]
sns.barplot(
    data=size_df,
    x="order_size",
    y="late_percentage",
    order=size_order,
    ax=axes[0, 1]
)
axes[0, 1].set_title("Late Delivery by Order Size")
axes[0, 1].set_xlabel("Order Size")
axes[0, 1].set_ylabel("Late Delivery (%)")

# 3. Late delivery by freight category
freight_order = ["Low (<20)", "Medium (20-50)", "High (50+)"]
sns.barplot(
    data=freight_df,
    x="freight_category",
    y="late_percentage",
    order=freight_order,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Late Delivery by Freight Cost")
axes[1, 0].set_xlabel("Freight Category")
axes[1, 0].set_ylabel("Late Delivery (%)")

# 4. Monthly late delivery trend
monthly_df["month"] = pd.to_datetime(monthly_df["month"])
sns.lineplot(
    data=monthly_df,
    x="month",
    y="late_percentage",
    marker="o",
    ax=axes[1, 1]
)
axes[1, 1].set_title("Monthly Late Delivery Trend")
axes[1, 1].set_xlabel("Purchase Month")
axes[1, 1].set_ylabel("Late Delivery (%)")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.suptitle("Olist E-Commerce Delivery Analysis", fontsize=16)
plt.tight_layout()

# Save the dashboard in the project reports folder
os.makedirs(os.path.join(project_path, "reports"), exist_ok=True)
chart_path = os.path.join(project_path, "reports", "eda_summary.png")
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Dashboard saved to:", chart_path)

In [ ]:

from scipy.stats import chi2_contingency

# Retrieve state and delivery outcome
state_data = pd.read_sql(
    text("""
        SELECT customer_state, is_late
        FROM analytics.order_level_base
    """),
    engine
)

# Create a contingency table
contingency_table = pd.crosstab(
    state_data["customer_state"],
    state_data["is_late"]
)

# Perform Chi-square test
chi2, p_value, degrees_of_freedom, expected = (
    chi2_contingency(contingency_table)
)

print("Chi-square statistic:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", degrees_of_freedom)

if p_value < 0.05:
    print("\nResult: Reject the null hypothesis.")
    print("Customer state is significantly associated with late delivery.")
else:
    print("\nResult: Fail to reject the null hypothesis.")
    print("No statistically significant association was detected.")


In [ ]:
query = """
SELECT *
FROM analytics.order_level_base
LIMIT 5;
"""

with engine.connect() as connection:
    model_data = pd.read_sql(text(query), connection)

display(model_data.dtypes)

In [ ]:

# Load the complete order-level dataset

query = """
SELECT *
FROM analytics.order_level_base
"""

with engine.connect() as connection:
    model_data = pd.read_sql(text(query), connection)

print("Dataset shape:", model_data.shape)
display(model_data.head())


In [ ]:
import numpy as np

# Convert purchase timestamp into useful features
model_data["purchase_month"] = (
    model_data["purchase_timestamp"].dt.month
)
model_data["purchase_day_of_week"] = (
    model_data["purchase_timestamp"].dt.dayofweek
)
model_data["purchase_hour"] = (
    model_data["purchase_timestamp"].dt.hour
)

# Freight cost relative to product value
model_data["freight_to_value_ratio"] = (
    model_data["total_freight"]
    / model_data["total_product_value"].replace(0, np.nan)
)

# Create the final prediction dataset
feature_columns = [
    "customer_state",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "total_items",
    "unique_sellers",
    "total_product_value",
    "total_freight",
    "freight_to_value_ratio",
    "average_item_price",
    "max_installments"
]

X = model_data[feature_columns].copy()
y = model_data["is_late"].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nMissing values:")
print(X.isnull().sum())
print("\nTarget distribution:")
print(y.value_counts())

In [ ]:

# Compare the stored target with the actual delivery timestamps

timestamp_late = (
    model_data["actual_delivery_timestamp"] >
    model_data["estimated_delivery_timestamp"]
).astype(int)

print("Stored is_late distribution:")
print(model_data["is_late"].value_counts().sort_index())

print("\nTimestamp-based distribution:")
print(timestamp_late.value_counts().sort_index())

print("\nNumber of mismatched labels:")
print((model_data["is_late"] != timestamp_late).sum())


In [ ]:
mismatches = model_data[
    model_data["is_late"] != timestamp_late
].copy()

mismatches["timestamp_based_label"] = timestamp_late.loc[
    mismatches.index
]

print("Total mismatches:", len(mismatches))

display(
    mismatches[
        [
            "order_id",
            "actual_delivery_timestamp",
            "estimated_delivery_timestamp",
            "is_late",
            "timestamp_based_label"
        ]
    ].head(15)
)

In [ ]:
# Compare calendar dates only
timestamp_late = (
    model_data["actual_delivery_timestamp"].dt.date >
    model_data["estimated_delivery_timestamp"].dt.date
).astype(int)

print("Stored is_late distribution:")
print(model_data["is_late"].value_counts().sort_index())

print("\nCalendar-date-based distribution:")
print(timestamp_late.value_counts().sort_index())

print("\nNumber of mismatches:")
print((model_data["is_late"] != timestamp_late).sum())

In [ ]:
# Calculate the promised delivery duration
model_data["promised_delivery_days"] = (
    model_data["estimated_delivery_timestamp"] -
    model_data["purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

# Check the new feature
print("Promised delivery days:")
display(model_data["promised_delivery_days"].describe())

print(
    "\nMissing values:",
    model_data["promised_delivery_days"].isna().sum()
)

In [ ]:

from sklearn.model_selection import train_test_split

# Final feature columns
feature_columns = [
    "customer_state",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "total_items",
    "unique_sellers",
    "total_product_value",
    "total_freight",
    "freight_to_value_ratio",
    "average_item_price",
    "max_installments",
    "promised_delivery_days"
]

# Prepare features and target
X = model_data[feature_columns].copy()
y = model_data["is_late"].copy()

# Sort chronologically
sorted_indices = model_data["purchase_timestamp"].sort_values().index
X = X.loc[sorted_indices]
y = y.loc[sorted_indices]

# Chronological 80/20 split
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

# Check the split
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining late percentage:", round(y_train.mean() * 100, 2), "%")
print("Testing late percentage:", round(y_test.mean() * 100, 2), "%")

print("\nTraining date range:")
print(model_data.loc[X_train.index, "purchase_timestamp"].min(),
      "to",
      model_data.loc[X_train.index, "purchase_timestamp"].max())

print("\nTesting date range:")
print(model_data.loc[X_test.index, "purchase_timestamp"].min(),
      "to",
      model_data.loc[X_test.index, "purchase_timestamp"].max())


In [ ]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create baseline model
dummy_model = DummyClassifier(strategy="most_frequent")

# Train
dummy_model.fit(X_train, y_train)

# Predict on test data
dummy_predictions = dummy_model.predict(X_test)

# Evaluate
print("Dummy Baseline Accuracy:",
      round(accuracy_score(y_test, dummy_predictions), 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    dummy_predictions,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, dummy_predictions))


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Separate categorical and numerical columns
categorical_features = ["customer_state"]

numerical_features = [
    col for col in feature_columns
    if col not in categorical_features
]

# Preprocess categorical features
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Preprocess numerical features
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, categorical_features),
    ("num", numerical_transformer, numerical_features)
])

# Create complete model pipeline
lr_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train model
lr_model.fit(X_train, y_train)

# Predictions
lr_predictions = lr_model.predict(X_test)
lr_probabilities = lr_model.predict_proba(X_test)[:, 1]

# Evaluation
print("LOGISTIC REGRESSION RESULTS")
print("=" * 40)

print("Accuracy:", round(accuracy_score(y_test, lr_predictions), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, lr_probabilities), 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    lr_predictions,
    target_names=["On-time", "Late"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_predictions))


In [ ]:

from sklearn.ensemble import RandomForestClassifier

# Create Random Forest pipeline
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        max_depth=15,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

# Train
rf_model.fit(X_train, y_train)

# Predictions
rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
print("RANDOM FOREST RESULTS")
print("=" * 40)

print("Accuracy:", round(accuracy_score(y_test, rf_predictions), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_probabilities), 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    rf_predictions,
    target_names=["On-time", "Late"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_predictions))


In [ ]:

from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

# Split training data chronologically for threshold selection
threshold_split = int(len(X_train) * 0.80)

X_subtrain = X_train.iloc[:threshold_split]
y_subtrain = y_train.iloc[:threshold_split]

X_threshold_val = X_train.iloc[threshold_split:]
y_threshold_val = y_train.iloc[threshold_split:]

# Train Logistic Regression on the earlier training portion
threshold_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

threshold_model.fit(X_subtrain, y_subtrain)

# Get validation probabilities
val_probabilities = threshold_model.predict_proba(
    X_threshold_val
)[:, 1]

# Find threshold that maximizes validation F1
precision, recall, thresholds = precision_recall_curve(
    y_threshold_val,
    val_probabilities
)

f1_scores = (
    2 * precision[:-1] * recall[:-1] /
    (precision[:-1] + recall[:-1] + 1e-10)
)

best_index = np.argmax(f1_scores)
best_threshold = thresholds[best_index]

print("Selected threshold:", round(best_threshold, 4))
print("Validation F1:", round(f1_scores[best_index], 4))

# Retrain on the complete training set
lr_model.fit(X_train, y_train)

# Evaluate on untouched test data using selected threshold
test_probabilities = lr_model.predict_proba(X_test)[:, 1]
tuned_predictions = (
    test_probabilities >= best_threshold
).astype(int)

print("\nTUNED LOGISTIC REGRESSION")
print("=" * 40)
print("Accuracy:", round(accuracy_score(y_test, tuned_predictions), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_probabilities), 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    tuned_predictions,
    target_names=["On-time", "Late"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, tuned_predictions))


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Get transformed feature names from preprocessing
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

# Extract Random Forest feature importance
importance_values = (
    rf_model.named_steps["classifier"].feature_importances_
)

# Create importance table
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
}).sort_values("Importance", ascending=False)

# Display top 10
print("TOP 10 IMPORTANT FEATURES")
display(importance_df.head(10))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(
    data=importance_df.head(10),
    x="Importance",
    y="Feature"
)
plt.title("Top 10 Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:

import os

project_path = "/content/drive/MyDrive/olist-late-delivery-prediction"

for root, dirs, files in os.walk(project_path):
    level = root.replace(project_path, "").count(os.sep)
    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")


In [ ]:

import os

project_path = "/content/drive/MyDrive/olist-late-delivery-prediction"

# Create the required folders
folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "sql",
    "reports",
    "models",
    "src"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folders created successfully!")

# Search Google Drive for your notebook
print("\nSearching for your project notebook...")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file == "Olist_Late_Delivery_Prediction.ipynb":
            print(os.path.join(root, file))


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith(".ipynb") and "Olist" in file:
            print(os.path.join(root, file))